In [ ]:
import pandas as pd
import numpy as np
from pyomo.environ import *
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
file_name="data/raw/early_2012_2013_loan_sample_with_outcome.csv"

In [ ]:
df=pd.read_csv(file_name)

In [ ]:
# check for duplicates
# count the total number of exact duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")
# duplicate rows (to inspect them)
if duplicate_count > 0:
    duplicate_rows = df[df.duplicated(keep=False)].sort_values(by=list(df.columns[:3]))
    print("\n--- Sample of Duplicate Rows ---")
    print(duplicate_rows.head(6))

In [ ]:
# check data structure
print("\nData Structure:")
df.info()

In [ ]:
# the dataset has 50000 records and 57 variables

In [ ]:
# view the first row
print(df.head(1).T)

In [ ]:
# drop id and member_id as they are identifiers adding no meaningful info

# drop variables describe what happened after the loan was already approved:
## funded_amnt, funded_amnt_inv represent the amount actually committed by the bank or investors
## issue_d represents the month and year the loan was funded
## out_prncp, out_prncp_inv: remaining outstanding principal >> direct result (the amount has not yet been paid back) of the loan's outcome (final status of the loan)
## total_pymnt, total_pymnt_inv: payments received to date; total_rec_prncp, total_rec_int, total_rec_late_fee: components of total payment received. These variables tell how much money was recovered. If they're used to predict default, the model will learn that people who paid $0 defaulted, which is useless for a new application
## recoveries, collection_recovery_fee: these variables only exist if a loan has already defaulted
## last_pymnt_d, last_pymnt_amnt, next_pymnt_d only exists for loans that have already been issued
## last_credit_pull_d often captures the pulls that happen after the loan was already in trouble. It is a symptom of the loan's outcome, not a predictor of its beginning

df = df.drop(columns = ['id', 'member_id',
                        'funded_amnt', 'funded_amnt_inv','issue_d',
                       'out_prncp', 'out_prncp_inv',
                       'total_pymnt', 'total_pymnt_inv','total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                       'recoveries', 'collection_recovery_fee',
                       'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
                       'last_credit_pull_d'])
# verify changes
print("Remaining columns:", df.shape[1])

In [ ]:
# data types
# loan_amnt, revol_bal indicates amount of money
# convert these variables to float
money_col = ['loan_amnt', 'revol_bal']
df[money_col]=df[money_col].astype(float)
# verify change
print(df[money_col].info())

# remove % from revol_util and covert the variable to float
df['revol_util'] = df['revol_util'].astype('string').str.strip('%').astype(float)
# verify change
print(df['revol_util'].info())

In [ ]:
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# summary statistics - object variables
obj_cols = df.select_dtypes(include=['object']).columns

# count of unique values
obj_unique_counts = df[obj_cols].nunique()

# missing Rate (%)
# .isnull().mean() gives the decimal, * 100 makes it a percentage
obj_missing_rates = (df[obj_cols].isnull().mean() * 100).round(2)

# summary table
obj_audit = pd.DataFrame({
    'Unique Values': obj_unique_counts,
    'Missing Rate (%)': obj_missing_rates
})

# sort by Unique Values to see "High Cardinality" at the top
obj_audit = obj_audit.sort_values(by='Unique Values', ascending=False)

print("--- Object Variables ---")
print(obj_audit)

In [ ]:
# title, purpose, and desc respectively are loan category, loan title and loan description provided by the borrower
# title is redundant with purpose. For the same purpose of debt_consolidation, the title has some variation (Debt consolidation, Debt Consolidation, debt consolidation, Consolidation, consolidation)
# crosstab btw title and purpose
cross_check = pd.crosstab(df['purpose'], df['title'])
# print top 10 most frequent combinations
print(cross_check.stack().nlargest(10))
# desc almost 40% of desc is missing, the non-missing observations have over 30K unique values which are unstructured free-form texts. Desc is highly subjective; borrowers may describe their situations in ways that is favourable to them
# desc and title create noises, drop these variables

In [ ]:
# zip_code and emp_title contain hundres/thousands of unique values which will introduce excessive noise
# regional risk can be captured by addr_state since states do have different default rates due to local economic conditions 
# emp_title is a free text field > difficult to categorise the values into meaningful groups. the dataset has annual_inc and emp_length which capture the economic value of the job title
#> drop these variables to prevent dimensionality explosion

In [ ]:
# application_type has only 1 value

In [ ]:
# focus on home_ownership, pymnt_plan, initial_list_status
target_obj_cols = ['home_ownership','pymnt_plan', 'initial_list_status']
baseline = df['loan_is_bad'].mean()

for var in target_obj_cols:
    # 1. Group and calculate raw stats
    stats = df.groupby(var).agg(
        count=(var, 'count'),
        default_rate=('loan_is_bad', 'mean')
    ).reset_index()
    
    # 2. Calculate the specific columns you requested
    stats['%dataset'] = (stats['count'] / len(df)) * 100
    stats['risk_lift'] = stats['default_rate'] / baseline
    
    # 3. Rename the 'var' column to 'variable' and store the category name
    stats = stats.rename(columns={var: 'category'})
    
    # 4. Reorder columns as requested
    stats = stats[['category', 'count', '%dataset', 'default_rate', 'risk_lift']]
    
    # 5. Clean up the formatting for printing
    print(f"\nData distribution for {var}:")
    print(stats.round({
        '%dataset': 2, 
        'default_rate': 2, 
        'risk_lift': 2
    }).to_string(index=False))

In [ ]:
# pymnt_plan has only 3 y cases >> insufficient >> drop
# initial_list_status: fractionally-funded loans are expected to have higher default rates; however, for the given dataset, both categories had the same default rates, indicating that funding mechanics are independent of borrower creditworthiness >> drop
# drop emp_title, desc, title, zip_code, application_type, pymnt_plan and initial_list_status
df = df.drop(columns = ['emp_title','desc', 'title','zip_code','application_type','pymnt_plan','initial_list_status'])
# verify changes
print("Remaining columns:", df.shape[1])

In [ ]:
# home_ownership: OTHER and NONE, each represents less than 0.1% of your data, making them statistically "silent"
# however, dropping these categories might lose potential bad loan signals
# merge OTHER and NONE into one category - OTHER
df['home_ownership'] = df['home_ownership'].replace('NONE', 'OTHER')
# verify change
ownership_percent = df['home_ownership'].value_counts(normalize=True) * 100
default_across_ownership = df.groupby('home_ownership')['loan_is_bad'].mean() * 100
ownership_summary = pd.DataFrame({
    'Percentage (%)': ownership_percent,
    'Default rate (%)': default_across_ownership
})
ownership_summary = ownership_summary.sort_values(by='Percentage (%)', ascending=False)
print("--- Home Ownership Distribution ---")
print(ownership_summary.round(2))

In [ ]:
# loan_status distribution
loan_status_counts = df['loan_status'].value_counts()
loan_status_percent = df['loan_status'].value_counts(normalize=True) * 100
default_across_loan_status = df.groupby('loan_status')['loan_is_bad'].mean() * 100
loan_status_summary = pd.DataFrame({
    'Count': loan_status_counts,
    'Percentage (%)': loan_status_percent,
    'Default rate (%)': default_across_loan_status
})

loan_status_summary = loan_status_summary.sort_values(by='Percentage (%)', ascending=False)
print("--- Loan Status Distribution ---")
print(loan_status_summary.round(2))

# current loans are currently classified as healthy (default rate at 0%)
# however, current loans represent a transient state with no observed outcome
# their inclusion in the dataset would introduce bias and distort evaluation of loan review policeis and predictive model performance
# drop these observations

# generally, loans marked as late or in grace period are probabilistic (in-progress status). Some borrowers may pay their loans while others will eventually default.
# in the given dataset, 100% of these loans are already labeled as bad. hence, they provide critical discriminatory power
# keep these observations

# drop observations where loan_status is current
status_to_drop = ['Current']
df = df[~df['loan_status'].isin(status_to_drop)].copy()
# verify change
print(f"New dataset size: {len(df)}")

In [ ]:
# summary statistics - numeric variables
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# unique, missing values
num_unique_counts = df[num_cols].nunique().sort_values(ascending=False)
num_missing_rates = (df[num_cols].isnull().mean() * 100).round(2)
num_audit = pd.DataFrame({
    'Unique Values': num_unique_counts,
    'Missing Rate (%)': num_missing_rates
})
num_audit = num_audit.sort_values(by='Unique Values', ascending=False)
print("--- Object Variables ---")
print(num_audit)

# outlier
outlier_summary = []

for var in num_cols: 
    # 1. 99th percentile
    q99 = df[var].quantile(0.99)
    
    # 2. Max value
    max_val = df[var].max()
    
    # 3. Calculate Skew Ratio (Max / 99th)
    # We use a conditional to avoid dividing by zero if q99 is 0
    skew_ratio = max_val / q99 if q99 != 0 else 0
    
    # 4. Identify extreme observations (above 99th)
    outliers_df = df[df[var] > q99]
    count = len(outliers_df)
    
    # 5. Default rate for these extreme observations
    if count > 0:
        # Assuming 'loan_is_bad' is already your binary target
        extreme_default_rate = outliers_df['loan_is_bad'].mean() * 100
    else:
        extreme_default_rate = 0

    # 6. Append everything to the list
    outlier_summary.append({
        'variable': var,
        '99th': round(q99, 2),
        'max': round(max_val, 2),
        'skew_ratio': round(skew_ratio, 2),
        'rows_above_99th': count,
        'extreme_default_rate(%)': round(extreme_default_rate, 2)
    })

# 7. Convert to DataFrame for easy viewing
summary_df = pd.DataFrame(outlier_summary)

# 8. Sort by Skew Ratio to see the "Longest Tails" at the top
print("--- Data Distribution & Outlier Risk Audit ---")
print(summary_df.sort_values(by='skew_ratio', ascending=False))

# 9. Overall baseline for comparison
overall_default_rate = df['loan_is_bad'].mean() * 100
print(f"\nOverall Default Rate (Baseline): {overall_default_rate:.2f}%")

# the difference btw the overall default rate and the default rate among extreme observations is small
# this suggests extreme behaviour does not necessarily lead to default

In [ ]:
# policy_code has only 1 value
# over 80% of mths_since_last_record, mths_since_last_major_derog is missing
# drop mths_since_last_record, mths_since_last_major_derog, policy_code
df = df.drop(columns = ['mths_since_last_record', 'mths_since_last_major_derog','policy_code'])
# verify changes
print("Remaining columns:", df.shape[1])

In [ ]:
# mths_since_last_delinq has 56% missing values
# A null here means the borrower has never defaulted. By filling this with a high number, you tell the model that "Never" is mathematically very far from "1 month ago."
# fill the specific column with 999
# This represents "Pristine History" (Never Delinquent)
df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(999)
# verify change
print(f"Remaining nulls in mths_since_last_delinq: {df['mths_since_last_delinq'].isnull().sum()}")

In [ ]:
# the number of missing values across variables related to revolving accounts are inconsistent
# total_credit_rv has ~30% missing values, revol_util 0.06% while revol_bal none
# investigate possible mathmatically illogical combinations of {revol_bal, total_credit_rv, revol_util}
# revol_util acts as one source of truth for creditworthiness
# clean revol_util there's no missing values left
# then drop total_credit_rv and revol_bal as their predictive value is already captured in revol_util
# dropping these variables also resolves the missingness issue in total_credit_rv without the need for potentially bias imputation

# scenario 1: revol_bal > 0; total_credit_rv >0; revol_util is either missing or equal to 0
revol_1 = df[(df['revol_bal']>0) & (df['total_credit_rv']>0) & (df['revol_util'].isna() | (df['revol_util']==0))]
print(f"Number of observations: {len(revol_1)}")
print(revol_1[['revol_bal', 'total_credit_rv','revol_util','loan_status', 'loan_is_bad']])
# calculate and fill the missing revol_util for these observations
# revol_util = revol_bal / total_credit_rv * 100%
df.loc[revol_1.index, 'revol_util'] = (revol_1['revol_bal']/revol_1['total_credit_rv'])*100
# Verify change
print(f"Recalculated values:") 
print(df.loc[revol_1.index, ['revol_bal', 'total_credit_rv', 'revol_util']])

In [ ]:
# scenario 2: revol_bal > 0; total_credit_rv = 0; revol_util is either missing or equal to or greater than 0
revol_2 = df[(df['revol_bal']>0) & (df['total_credit_rv']==0)]
print(f"Number of observations: {len(revol_2)}")

In [ ]:
# scenario 3: revol_bal > 0; total_credit_rv is missing; revol_util is either missing or equal to 0
revol_3 = df[(df['revol_bal']>0) & df['total_credit_rv'].isna() & (df['revol_util'].isna() | (df['revol_util']==0))]
print(f"Number of observations: {len(revol_3)}")
print(revol_3[['revol_bal', 'total_credit_rv','revol_util','loan_status', 'loan_is_bad']])
# cannot guess the value of revol_util. These observations create noises
# since these observations are statistically insignificant, drop them
df = df.drop(revol_3.index)
# verify change
print(f"New dataset size: {len(df)}")

In [ ]:
# scenario 4: revol_bal is either missing or equal to zero, total_credit_rv is equal to zero
revol_4 = df[(df['revol_bal'].isna() | df['revol_bal']==0) & (df['total_credit_rv']==0)]
print(f"Number of observations: {len(revol_4)}")
print(revol_4[['revol_bal', 'total_credit_rv','revol_util','loan_status', 'loan_is_bad']])
# converting the revol_util values of these observations to zero makes "No Credit" the same as "Perfect Credit"
# in reality, someone with no history holds a higher risk than someone with a record of 0% revolving utilisation
# drop these observations as they create noises and are statistically insignificant
df = df.drop(revol_4.index)
# verify change
print(f"New dataset size: {len(df)}")

In [ ]:
# scenario 5: revol_bal = 0; both total_credit_rv and revol_util are either greater than 0 or missing
revol_5 = df[(df['revol_bal'] == 0) & (df['total_credit_rv'].isna() | (df['total_credit_rv']>0)) & (df['revol_util'].isna() | (df['revol_util']>0))]
print(f"Number of observations: {len(revol_5)}")
print(revol_5[['revol_bal', 'total_credit_rv','revol_util','loan_status', 'loan_is_bad']])
# calculate and fill the missing revol_util for these observations
# revol_util = 0 / total_credit_rv * 100% = 0
df.loc[revol_5.index, 'revol_util'] = 0
# verify change
print(f"Recalculated values:") 
print(df.loc[revol_5.index, ['revol_bal', 'total_credit_rv', 'revol_util']])

In [ ]:
# scenario 6: revol_bal is missing; total_credit_rv > 0, revol_util is missing
revol_6 = df[df['revol_bal'].isna() & (df['total_credit_rv']>0) & df['total_credit_rv'].isna()]
print(f"Number of observations: {len(revol_6)}")

In [ ]:
# scenario 7: both revol_bal and total_credit_rv are missing, but revol_util is equal to or greater than 0
revol_7 = df[df['revol_bal'].isna() & df['total_credit_rv'].isna() & df['total_credit_rv'].notna()]
print(f"Number of observations: {len(revol_7)}")

In [ ]:
# scenario 8: all revol_bal, total_credit_rv and revol_util are missing
revol_8= df[df['revol_util'].isna() & df['revol_bal'].isna() & df['total_credit_rv'].isna()]
print(f"Number of observations: {len(revol_8)}")

In [ ]:
# verify changes to revol_util
count_missing_util = df[df['revol_util'].isna()]
print(f"Number of missing values in revol_util: {len(count_missing_util)}")

In [ ]:
# drop revol_bal and total_credit_rv
df = df.drop(columns = ['revol_bal','total_credit_rv'])
# verify change
print("Remaining columns:", df.shape[1])

In [ ]:
# emp_length: 3% missing. impute missing values with median - the neutral middle ground that won't pull clusters to either extreme (newbie vs veteran)
# calculate the median employment length
emp_median = df['emp_length'].median()
# fill the 3% missing values
df['emp_length'] = df['emp_length'].fillna(emp_median)
# verify the fix
print(f"Imputed 'emp_length' with Median: {emp_median} years")
print(f"Remaining nulls: {df['emp_length'].isnull().sum()}")

In [ ]:
# tot_coll_amt is extremely skewed
# 99% of all borrowers have a total collection amount ever owed of 1K or less, but some have  annually
# investigate observations with extreme tot_coll_amt values
top_1_percent_coll = df['tot_coll_amt'].quantile(0.99)
extreme_coll_df = df[df['tot_coll_amt'] > top_1_percent_coll][
    ['annual_inc', 'dti', 'revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad']
]
# sort by collection amount descending to see observation with the most extreme value first
extreme_coll_df = extreme_coll_df.sort_values(by='tot_coll_amt', ascending=False)
print(f"Number of observations with significant amount of collection: {len(extreme_coll_df)}")
print(extreme_coll_df.head(5))

# show observation where loan is good
extreme_coll_default = df[
    (df['tot_coll_amt'] >= top_1_percent_coll) & 
    (df['loan_is_bad'] == False)][['annual_inc', 'dti','revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad'
]]
extreme_coll_default_df = extreme_coll_default.sort_values(by='tot_coll_amt', ascending=False)
print(f"Number of observations with significant amount of collection but loan is good: {len(extreme_coll_default_df)}")
print(extreme_coll_default_df.head(5))

# percentage of bad loans
print(f"Bad Loan Rate for Top 1%: {df[df['tot_coll_amt'] >= top_1_percent_coll]['loan_is_bad'].mean():.2%}")

# tot_coll_amt represents failed obligations
# generally, greater amount of collection means a borrower is more likley to default as it indicates a pattern of systemic financial distress or a blatant disregard for repayment 
# however, among borrowers with most collection, some still fulfilled their repayment obligations
# among these observations, the default rate is 21%
# keep these observations

In [ ]:
# tot_coll_amt has ~30% missing values
# the variable is extremely skewed, skewed_rate = 54.87%
# tot_coll_amt represents the total amount the borrower ever owed in collections
# if missing, it's likely that the borrower has zero collections
# impute missing values with 0
df['tot_coll_amt'] = df['tot_coll_amt'].fillna(0)

In [ ]:
# tot_cur_bal moderately skewed (skew rate = 12.43%)
# investigate observations with extreme tot_cur_bal values
top_1_percent_curbal = df['tot_cur_bal'].quantile(0.99)
extreme_curbal_df = df[df['tot_cur_bal'] > top_1_percent_curbal][
    ['annual_inc', 'dti', 'revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad']
]
# sort by total revolving credit limit descending to see observation with the most extreme value first
extreme_curbal_df = extreme_curbal_df.sort_values(by='tot_cur_bal', ascending=False)
print(f"Number of observations with current balance greater than 99th quantile level: {len(extreme_curbal_df)}")
print(extreme_curbal_df.head(5))

# show observation where loan is good
extreme_curbal_default = df[
    (df['tot_cur_bal'] >= top_1_percent_curbal) & 
    (df['loan_is_bad'] == False)][['annual_inc', 'dti', 'revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad'
]]
extreme_curbal_default_df = extreme_curbal_default.sort_values(by='tot_cur_bal', ascending=False)
print(f"Number of observations with significant current balance but loan is good: {len(extreme_curbal_default_df)}")
print(extreme_curbal_default_df.head(5))

# percentage of bad loans
print(f"Bad Loan Rate for Top 1%: {df[df['tot_cur_bal'] >= top_1_percent_curbal]['loan_is_bad'].mean():.2%}")

# higher current balance = more debts = more likely to default
# among top borrowers with most current balance, the default rate is 10%
# keep these observations

In [ ]:
# tot_cur_bal also has ~30% missing values
# dropping observations with missing values would losing too much information
# imputing missing values with mean would pull this indicator upward
# median imputation is the safest approach

# 30% of borrowers didnt reprt their total balance; this might be a specific risk group
# create a flag: 1 if data was missing, 0 if it was provided
df['tot_cur_bal_missing'] = df['tot_cur_bal'].isnull().astype(int)

# calculate the median
median_bal = df['tot_cur_bal'].median()
# fill the missing values
df['tot_cur_bal'] = df['tot_cur_bal'].fillna(median_bal)
print(f"Imputing missing 'tot_cur_bal' with Median: {median_bal:,.2f}")
#  verify change
print(f"Remaining nulls in tot_cur_bal: {df['tot_cur_bal'].isnull().sum()}")

In [ ]:
# annual_inc: 99% of all borrowers earn an annual income of 225K or less, but some earn up to 55KM annually
# investigate observations with extreme annual_inc values
top_1_percent_inc = df['annual_inc'].quantile(0.99)
extreme_income_df = df[df['annual_inc'] > top_1_percent_inc][
    ['annual_inc', 'dti','revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad']
]
# sort by income descending to see observation with the most extreme value first
extreme_income_df = extreme_income_df.sort_values(by='annual_inc', ascending=False)
print(f"Number of observations with income greater than 99th quantile value: {len(extreme_income_df)}")
print(extreme_income_df.head(5))

# show observation where loan is bad
extreme_income_default = df[
    (df['annual_inc'] >= top_1_percent_inc) & 
    (df['loan_is_bad'] == True)][['annual_inc', 'dti', 'revol_util', 'tot_cur_bal', 'tot_coll_amt', 'loan_status', 'loan_is_bad'
]]
extreme_income_default_df = extreme_income_default.sort_values(by='annual_inc', ascending=False)
print(f"Number of observations with income greater than that level but loan is bad: {len(extreme_income_default_df)}")
print(extreme_income_default_df.head(5))

# percentage of bad loans among top earners
print(f"Bad Loan Rate for Top 1% Earners: {df[df['annual_inc'] >= top_1_percent_inc]['loan_is_bad'].mean():.2%}")

# there are 494 observations with extreme income values
# some borrowers had a revol_util rate over 90% but they maintained very low dti ratio (1-2%) and also fully paid their debts. 
# however, some borrowers appeared creditworthy but still failed to repay their loans
# among the top 1% earners, the default rate is 10%. This indicates that income standalone lacks discriminatory power to predict default risk
# these observations are retained to ensure the model captures the nuances of high-income borrower behaviour

In [ ]:
# verify number of missing values
remain_num_cols = df.select_dtypes(include=['int64', 'float64']).columns
remain_num_unique_counts = df[remain_num_cols].nunique().sort_values(ascending=False)
remain_num_missing_rates = (df[remain_num_cols].isnull().mean() * 100).round(2)
remain_num_audit = pd.DataFrame({
    'Unique Values': remain_num_unique_counts,
    'Missing Rate (%)': remain_num_missing_rates
})
remain_num_audit = remain_num_audit.sort_values(by='Unique Values', ascending=False)
print("--- Object Variables ---")
print(remain_num_audit)

In [ ]:
# check for illogical installment
r = (df['int_rate'] / 100) / 12
n = df['term']
pv = df['loan_amnt']
# calculate installment
df['calc_installment'] = (r * pv) / (1 - (1 + r)**-n)
# check the difference
df['inst_diff'] = np.abs(df['calc_installment'] - df['installment'])
#number of rows where the difference is more than £1 (allowing for small rounding)
illogical_rows = df[df['inst_diff'] > 1.0]
print(f"Number of illogical installments found: {len(illogical_rows)}")

In [ ]:
# calculate the percentage error
df['error_percentage'] = (df['inst_diff'] / df['installment']) * 100
# summary of the errors
print(df['error_percentage'].describe())
# number of rows with more than 5% difference
major_errors = df[df['error_percentage'] > 5.0]
print(f"Number of rows with >5% error: {len(major_errors)}")

In [ ]:
# drop observations with significant error (>5% difference)
# installment is redundant since it is derivative of loan amount, interest and term
# drop installment and columns created to check illogicality in installment
df = df[df['error_percentage'] <= 5.0].copy()
df = df.drop(columns=['installment', 'calc_installment', 'inst_diff', 'error_percentage'])
# verify change
print("Remaining columns:", df.shape[1])
print(f"New dataset size: {len(df)}")

In [ ]:
# check whether there're any observations with open_acc greater than total_acc
logic_violation = df[df['open_acc'] > df['total_acc']]
print(f"Number of logical violations (open_acc > total_acc): {len(logic_violation)}")
if len(logic_violation) > 0:
    print(logic_violation[['open_acc', 'total_acc']])

In [ ]:
# check for multicollinearity
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
# correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Financial Features")
plt.show()

In [ ]:
# total_acc and open_acc is correlated. drop total_acc as open_acc reflects current behaviour of borrowers
df = df.drop(columns = ['total_acc'])
# verify change
print("Remaining columns:", df.shape[1])

In [ ]:
target_int_cols = ['pub_rec', 'delinq_2yrs', 'open_acc', 'inq_last_6mths']
baseline = df['loan_is_bad'].mean()

for var in target_int_cols:
    # 1. Group and calculate raw stats
    stats = df.groupby(var).agg(
        count=(var, 'count'),
        default_rate=('loan_is_bad', 'mean')
    ).reset_index()
    
    # 2. Calculate the specific columns you requested
    stats['%dataset'] = (stats['count'] / len(df)) * 100
    stats['risk_lift'] = stats['default_rate'] / baseline
    
    # 3. Rename the 'var' column to 'variable' and store the category name
    stats = stats.rename(columns={var: 'category'})
    
    # 4. Reorder columns as requested
    stats = stats[['category', 'count', '%dataset', 'default_rate', 'risk_lift']]
    
    # 5. Clean up the formatting for printing
    print(f"\nData distribution for {var}:")
    print(stats.round({
        '%dataset': 2, 
        'default_rate': 2, 
        'risk_lift': 2
    }).to_string(index=False))

In [ ]:
# cap extreme values
# 1. Define Continuous Monetary Variables (Use 99th Percentile)
monetary_vars = ['annual_inc', 'tot_cur_bal', 'tot_coll_amt']

# 2. Define Discrete Count Variables (Use Logical Risk Caps)
# These caps are based on your Risk Lift and Distribution tables
count_caps = {
    'pub_rec': 3,          # Risk stabilizes/counts drop after 3
    'delinq_2yrs': 3,      # High delinquency is rare and high risk
    'inq_last_6mths': 6,   # More than 6 inquiries is extreme behavior
    'open_acc': 35         # Use ~99th percentile for account counts
}

print("--- Applying Statistical & Logical Caps ---")

# Apply 99th Percentile to Monetary Vars
for var in monetary_vars:
    if var in df.columns:
        q99 = df[var].quantile(0.99)
        df[var] = df[var].clip(upper=q99)
        print(f"{var: <15} | Capped at 99th Percentile: {q99:,.2f}")

# Apply Manual Caps to Count Vars
for var, cap_value in count_caps.items():
    if var in df.columns:
        df[var] = df[var].clip(upper=cap_value)
        print(f"{var: <15} | Capped at Logical Limit: {cap_value}")

print("\nOutlier distortion removed. Ready for Scaling.")

In [ ]:
df.info()

In [ ]:
# Compare averages of key metrics across different Grades
grade_analysis = df.groupby('grade')[['dti','delinq_2yrs', 'int_rate', 'revol_util']].mean()

# Round for readability
print(grade_analysis.round(2))

In [ ]:
# dti ranges from 14.88% (grade A) to 19.96% (grade G)
# this narrow range suggests that loan grade isn's heavily punishing people for being over-leveraged.
# add a cap on DTI, regardless of grade

In [ ]:
# grade G has delinquencies four time higher than that of grade A
# grade already considers borrower's poor behaviour

In [ ]:
# big jump in int_rate btw grade A and grade B
# analyse default rate across sub-grades

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calculate the Default Rate for every Sub-Grade
subgrade_risk = df.groupby('sub_grade')['loan_is_bad'].mean().sort_index()

# 2. Plot the "Risk Ladder"
plt.figure(figsize=(15, 6))
sns.barplot(x=subgrade_risk.index, y=subgrade_risk.values, palette='viridis')
plt.title('Default Rate by Sub-Grade (The Risk Ladder)')
plt.ylabel('Proportion of Bad Loans')
plt.xlabel('Sub-Grade')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# default rate from B1 to B5 is relatively flat
# there's a massive jump btw C2 and C3
# default rate across grade C3 to D3 is relatively flat

In [ ]:
# Trial 1: Without DTI
# policy A: approve loan grade A-C
# policy B: approve loan grade A-D

# 1. Setup the Risk Ladder
subgrade_order = sorted(df['sub_grade'].unique())

# --- POLICY A: Approve ALL Grades A, B, and C ---
# A1 to C5 is 15 sub-grades (Indexes 0 to 14)
# We use :15 because the stop index is exclusive
policy_a_list = subgrade_order[:15] 
df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# --- POLICY B: Approve ALL Grades A, B, C, and D ---
# A1 to D5 is 20 sub-grades (Indexes 0 to 19)
# We use :20 to include up to D5
policy_b_list = subgrade_order[:20] 
df['approved_B'] = df['sub_grade'].isin(policy_b_list)

# 2. Business Logic Assumptions
LGD = 0.70  # Loss Given Default (Standard assumption for unsecured loans)

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    if len(approved) == 0:
        return {"Policy": name, "Default Rate (%)": 0, "Approved Count": 0, "Net Profit (£)": 0}
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Interest earned from people who paid back
    # (Loan Amount * Interest Rate / 100)
    good_loans = approved[approved['loan_is_bad'] == 0]
    revenue = (good_loans['loan_amnt'] * (good_loans['int_rate'] / 100)).sum()
    
    # Loss: Principal lost from people who defaulted
    # (Loan Amount * LGD)
    bad_loans = approved[approved['loan_is_bad'] == 1]
    losses = (bad_loans['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# 3. Run and Compare
summary = pd.DataFrame([
    evaluate_policy('Policy A (A-C)', df['approved_A']),
    evaluate_policy('Policy B (A-D)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)

print(summary)

In [ ]:
# Trial 2: Without DTI
# Policy A: approve loan grade A1-C2
# Policy B: approve loan grade A1-C5

policy_a_list = subgrade_order[:12]
df['approved_A'] = df['sub_grade'].isin(policy_a_list)

policy_b_list = subgrade_order[:15] 
df['approved_B'] = df['sub_grade'].isin(policy_b_list)

# 2. Business Logic Assumptions
LGD = 0.70  # Loss Given Default (Standard assumption for unsecured loans)

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    if len(approved) == 0:
        return {"Policy": name, "Default Rate (%)": 0, "Approved Count": 0, "Net Profit (£)": 0}
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Interest earned from people who paid back
    # (Loan Amount * Interest Rate / 100)
    good_loans = approved[approved['loan_is_bad'] == 0]
    revenue = (good_loans['loan_amnt'] * (good_loans['int_rate'] / 100)).sum()
    
    # Loss: Principal lost from people who defaulted
    # (Loan Amount * LGD)
    bad_loans = approved[approved['loan_is_bad'] == 1]
    losses = (bad_loans['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# 3. Run and Compare
summary = pd.DataFrame([
    evaluate_policy('Policy A (A-C)', df['approved_A']),
    evaluate_policy('Policy B (A-D)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)

print(summary)

In [ ]:
# Trial 3: Without DTI
# Policy A: approve loan grade A1-C2
# Policy B: approve loan grade A1-C3

policy_a_list = subgrade_order[:12]
df['approved_A'] = df['sub_grade'].isin(policy_a_list)

policy_b_list = subgrade_order[:13] 
df['approved_B'] = df['sub_grade'].isin(policy_b_list)

# 2. Business Logic Assumptions
LGD = 0.70  # Loss Given Default (Standard assumption for unsecured loans)

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    if len(approved) == 0:
        return {"Policy": name, "Default Rate (%)": 0, "Approved Count": 0, "Net Profit (£)": 0}
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Interest earned from people who paid back
    # (Loan Amount * Interest Rate / 100)
    good_loans = approved[approved['loan_is_bad'] == 0]
    revenue = (good_loans['loan_amnt'] * (good_loans['int_rate'] / 100)).sum()
    
    # Loss: Principal lost from people who defaulted
    # (Loan Amount * LGD)
    bad_loans = approved[approved['loan_is_bad'] == 1]
    losses = (bad_loans['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# 3. Run and Compare
summary = pd.DataFrame([
    evaluate_policy('Policy A (A1-C2)', df['approved_A']),
    evaluate_policy('Policy B (A1-C3)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)

print(summary)

In [ ]:
# Trial 4: with DTI
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of DTI < 30% for sub-grade C3-C5

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:15] # C3 to C5

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['dti'] < 30))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
# Trial 5: with DTI
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of DTI < 20% for sub-grade C3-C5

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:15] # C3 to C5

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['dti'] < 20))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
# Trial 6: with DTI
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of DTI < 10% for sub-grade C3-C5

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:15] # C3 to C5

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['dti'] < 10))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
# Trial 6: with DTI
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of DTI < 10% for sub-grade C3-C4

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:14] # C3 to C4

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['dti'] < 10))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
# Trial 7: with revol_util
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of revol_util for sub < 60% for grade C3-C5

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:15] # C3 to C5

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['revol_util'] < 60))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
# Trial 8: with revol_util
# policy A: approve loan grade A1-C2
# policy B: approve loan grade A1-C2, additional condition of revol_util for sub < 50% for grade C3-C5

# 1. Policy A: Conservative
policy_a_list = subgrade_order[:12] # A1 to C2

df['approved_A'] = df['sub_grade'].isin(policy_a_list)

# 2. Policy B: The Selective Growth (A1 to C2 + Conditional C3-C5)
tier_1 = subgrade_order[:12]  # A1 to C2
tier_2 = subgrade_order[12:15] # C3 to C5

df['approved_B'] = (df['sub_grade'].isin(tier_1)) | \
                   (df['sub_grade'].isin(tier_2) & (df['revol_util'] < 50))

LGD = 0.70 

def evaluate_policy(name, mask):
    approved = df[mask].copy()
    
    # Calculate Metrics
    vol = len(approved)
    def_rate = approved['loan_is_bad'].mean()
    
    # Revenue: Sum of (Loan Amount * Interest Rate) for 'Good' loans
    revenue = (approved[approved['loan_is_bad'] == 0]['loan_amnt'] * approved[approved['loan_is_bad'] == 0]['int_rate'] / 100).sum()
    
    # Loss: Sum of (Loan Amount * LGD) for 'Bad' loans
    losses = (approved[approved['loan_is_bad'] == 1]['loan_amnt'] * LGD).sum()
    
    net_profit = revenue - losses
    
    return {
        'Policy': name,
        'Default Rate (%)': round(def_rate * 100, 2),
        'Approved Count': vol,
        'Net Profit (£)': round(net_profit, 0)
    }

# Run the Comparison
summary = pd.DataFrame([
    evaluate_policy('Policy A (Conservative)', df['approved_A']),
    evaluate_policy('Policy B (Selective Growth)', df['approved_B'])
])

# Add a 'Profit Per Loan' column to see efficiency
summary['Profit Per Loan (£)'] = (summary['Net Profit (£)'] / summary['Approved Count']).round(2)


print(summary)

In [ ]:
#Through iterative policy testing, it was discovered that adding secondary behavioral filters (DTI < 30% or Revolving Utilization < 60%) to lower-tier grades (C3-D5) did not enhance net profitability. This suggests that the existing Grade assigned by the lender is an efficient aggregator of risk. The most profitable strategy was identified as a 'Hard Cut-off' at Sub-grade C2, which maximizes the Risk-Adjusted Return on Capital (RAROC) by avoiding the 'uncompensated risk' found in the C3-D5 segments

In [ ]:
# 1. Feature Engineering: Credit History Age
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])
df['credit_hist_years'] = 2026 - df['earliest_cr_line'].dt.year

# 2. Ordinal Mapping for Grade (Preserves the Risk Order)
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade_num'] = df['grade'].map(grade_map)

# 3. One-Hot Encoding for Nominal Categorical Vars
# We use drop_first=True to avoid the "Dummy Variable Trap" (Multicollinearity)
categorical_to_encode = ['home_ownership', 'verification_status', 'purpose']
df_encoded = pd.get_dummies(df[categorical_to_encode], drop_first=True)

# 4. Final Feature Selection for Clustering
# Combine your cleaned numeric features with the new encoded features
numeric_features = [
    'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'delinq_2yrs', 
    'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 
    'pub_rec', 'revol_util', 'tot_coll_amt', 'tot_cur_bal', 
    'emp_length', 'credit_hist_years', 'grade_num'
]

X = pd.concat([df[numeric_features], df_encoded], axis=1)

print(f"Total features ready for scaling: {X.shape[1]}")

In [ ]:
# standardisation for clustering
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Standard Scaler
# This will calculate the Mean and Standard Deviation for every column in X
scaler = StandardScaler()

# 2. Fit and Transform the combined feature set
# This converts every value into its Z-score: (x - mean) / std
X_scaled_values = scaler.fit_transform(X)

# 3. Reconstruct into a DataFrame
# It's vital to put the column names back so we can interpret the clusters later!
X_standardized = pd.DataFrame(X_scaled_values, columns=X.columns)

# 4. Final Verification
print("--- Normalization Summary ---")
print(f"Mean of first 5 features (should be ~0): \n{X_standardized.iloc[:, :5].mean().round(2)}")
print(f"\nStd Dev of first 5 features (should be ~1): \n{X_standardized.iloc[:, :5].std().round(2)}")

# Display the first few rows to ensure it looks correct
X_standardized.head()